# Orbital Debris Database: Building the Analytical SQLite Source
 
**Datasets:**
- kinetic_master.csv (cleaned, merged, and physics-enriched object list)
 
**Objective:** Create a normalized SQLite database from the kinetic master file, with tables designed for efficient queries and visualizations.
 
### Why this notebook?
The project needs a single, query-ready database that brings together all cleaned and derived orbital object data. This notebook takes the master CSV and builds a normalized SQLite database for analysis and visualization.
 
### What we do here
1. **Load the master dataset:** Read in the cleaned kinetic_master.csv file.
2. **Design the schema:** Decide on tables, primary keys, and relationships for efficient queries.
3. **Clean and patch metadata:** Standardize and fill in missing values, especially for ownership and launch details.
4. **Build and export tables:** Create normalized tables and write them to SQLite.
5. **Run validation checks:** Confirm data integrity and schema alignment.
 
This sets up the foundation for all downstream queries, charts, and risk modeling.

In [1]:
import pandas as pd
import sqlite3
import utility as utils

df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

### Stage 1.0: Standardize Ownership Fields
 
**The issue:**
- Owner codes and names in the master dataset have inconsistent formatting and naming conventions, especially for major operators like SpaceX.
- Inconsistent owner fields can cause join errors and reduce data quality.
 
**What we do:**
- Strip whitespace and standardize case for `owner_code` and `owner`.
- Map common variations of SpaceX and related names to a single canonical form.
 
**Why it matters:**
- Ensures all ownership fields are consistent and ready for reliable joins and grouping in downstream tables.

In [2]:
# Standardize owner_code and owner fields
df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

df['owner'] = df['owner'].replace(owner_name_map)

### Stage 1.1: Aggregate Ownership Metadata
 
**The issue:**
- Boolean sector flags (commercial, government, military, civil) may have mixed types or missing values, making analysis unreliable.
- Ownership metadata is spread across multiple rows and needs to be aggregated for normalization.
 
**What we do:**
- Coerce all flag columns to consistent 0/1 numeric types.
- Aggregate ownership metadata by `owner_code`, taking the first non-null value for string fields and the max for boolean flags.
- Create a unique, joinable `ownership_operators` table for the database.
 
**Why it matters:**
- Ensures sector flags are reliable for analysis and visualization.
- Aggregated ownership metadata enables efficient joins and reduces redundancy in the database.

In [3]:
# Coerce flag columns to 0/1 and aggregate ownership metadata
flag_cols = ['is_commercial', 'is_government', 'is_military', 'is_civil']

for col in flag_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    
owner_profile = df.groupby('owner_code', as_index=False).agg({
    'owner': utils.first_non_null,
    'country_operator': utils.first_non_null,
    'users': utils.first_non_null,
    'is_commercial': 'max',
    'is_government': 'max',
    'is_military': 'max',
    'is_civil': 'max',
    'contractor': utils.first_non_null,
    'contractor_country': utils.first_non_null
})

### Stage 1.2: Patch and Fill Key Metadata
 
**The issue:**
- Some fields (e.g., `primary_purpose`, `un_registry`, `lifetime_years`, `orbit_type`, `launch_id`) are missing or inconsistent, especially for non-payload objects.
- Incomplete or inconsistent metadata can cause errors in downstream analysis and reduce data quality.
 
**What we do:**
- For non-payload objects, fill missing `primary_purpose` and `un_registry` with 'Not Applicable'.
- Ensure `lifetime_years` is numeric and nullable (no forced imputation).
- Fill missing `orbit_type` with 'Other/Misc'.
- Synthesize `launch_id` from the COSPAR prefix (YYYY-NNN), filling missing values with 'UNKNOWN'.
 
**Why it matters:**
- Ensures all key fields are complete and consistent, supporting robust queries and analysis in the final database.

In [4]:
is_payload = df['object_type'].astype(str).str.strip().str.upper().eq('PAYLOAD')

df.loc[~is_payload, 'primary_purpose'] = df.loc[~is_payload, 'primary_purpose'].fillna('Not Applicable')
df.loc[~is_payload, 'un_registry'] = df.loc[~is_payload, 'un_registry'].fillna('Not Applicable')

df['lifetime_years'] = pd.to_numeric(df['lifetime_years'], errors='coerce')
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)
df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

### Stage 2.0: Build and Export Database Tables

**The issue:**
- The master DataFrame contains all orbital object data, but analysis and visualization require normalized, query-ready tables in SQLite.
- Without normalization, queries are slow, error-prone, and difficult to maintain.

**What we do:**
- Build individual DataFrames for each logical table (satellites, orbital data, ownership, launches, etc.).
- Export each DataFrame to SQLite with schema-aligned table names.
- Run validation checks to ensure data integrity and schema alignment.

**Why it matters:**
- Normalized tables enable efficient queries, reduce redundancy, and support robust analysis and visualization.
- Validation ensures the exported database is reliable for all downstream work.

In [5]:
# build the individual tables for SQLite export, selecting relevant columns and dropping duplicates where necessary.
df_ownership_operators = owner_profile[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
 ]

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])


df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'in_orbit',
     'owner_code', 'launch_id']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)

3939

### Stage 3: Sanity Checks and Validation

**The issue:**
- Exported tables may have missing keys, duplicates, or schema mismatches that can break downstream queries.

**What we do:**
- For each table, load from SQLite and run `utils.quick_report` to check for nulls, duplicates, and schema alignment.
- Commit and close the database connection after validation.

**Why it matters:**
- Ensures the exported database is reliable, complete, and ready for analysis and visualization.

In [6]:
primary_keys = {
    'satellites': 'norad_id',
    'orbital_data': 'norad_id',
    'ucs_details': 'norad_id',
    'risk_assessment': 'norad_id',
    'ownership_operators': 'owner_code',
    'launch_events': 'launch_id'
}

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)['name']

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    key_col = primary_keys.get(table)
    utils.quick_report(df, title=f"Table: {table}", key_col=key_col)

conn.commit()
conn.close()

print('\nSQLite build complete: ../data/clean/orbital_debris.db')

# Table: satellites

**Dimensions:** 33,358 rows × 12 columns

**Memory Footprint:** 16.10 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **cospar_id** | `object` | 0 | 100.0% | ✅ |
| **object_name** | `object` | 0 | 100.0% | ✅ |
| **satellite_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **official_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **object_type** | `object` | 0 | 100.0% | ✅ |
| **category** | `object` | 0 | 100.0% | ✅ |
| **ops_status** | `object` | 0 | 100.0% | ✅ |
| **data_status** | `object` | 32,414 | 2.8% | ⚠️ |
| **in_orbit** | `int64` | 0 | 100.0% | ✅ |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **launch_id** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|                |   count |   unique | top              |   freq |
|:---------------|--------:|---------:|:-----------------|-------:|
| cospar_id      |   33358 |    33358 | 1958-002B        |      1 |
| object_name    |   33358 |    18858 | FENGYUN 1C DEB   |   2340 |
| satellite_name |    5505 |     5492 | Starlink-2213    |      2 |
| official_name  |    5505 |     5483 | Jilin-1          |      5 |
| object_type    |   33358 |        4 | PAYLOAD          |  18339 |
| category       |   33358 |        5 | Active Satellite |  13106 |
| ops_status     |   33358 |        7 | UNKNOWN          |  16567 |
| data_status    |     944 |        2 | NEA              |    943 |
| owner_code     |   33358 |      105 | US               |  17083 |
| launch_id      |   33358 |     3939 | 1999-025         |   2346 |

### 📈 Numeric Overview
|          |   count |    mean |     std |   min |     25% |     50% |     75% |   max |
|:---------|--------:|--------:|--------:|------:|--------:|--------:|--------:|------:|
| norad_id |   33358 | 42670.6 | 18778.5 |     5 | 28912.2 | 44884.5 | 59505.8 | 68379 |
| in_orbit |   33358 |     1   |     0   |     1 |     1   |     1   |     1   |     1 |

# Table: orbital_data

**Dimensions:** 33,358 rows × 16 columns

**Memory Footprint:** 8.59 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **orbit_class** | `object` | 0 | 100.0% | ✅ |
| **orbit_type** | `object` | 0 | 100.0% | ✅ |
| **period_minutes** | `float64` | 0 | 100.0% | ✅ |
| **perigee_km** | `float64` | 0 | 100.0% | ✅ |
| **apogee_km** | `float64` | 0 | 100.0% | ✅ |
| **inclination_degrees** | `float64` | 0 | 100.0% | ✅ |
| **eccentricity** | `float64` | 0 | 100.0% | ✅ |
| **semi_major_axis_km** | `float64` | 0 | 100.0% | ✅ |
| **launch_mass_kg** | `float64` | 27,853 | 16.5% | ⚠️ |
| **proxy_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **dry_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **power_watts** | `float64` | 27,853 | 16.5% | ⚠️ |
| **proxy_power_watts** | `float64` | 0 | 100.0% | ✅ |
| **rcs** | `float64` | 0 | 100.0% | ✅ |
| **rcs_class** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| orbit_class |   33358 |        5 | LEO        |  27302 |
| orbit_type  |   33358 |        5 | Other/Misc |  28490 |
| rcs_class   |   33358 |        4 | UNKNOWN    |  18639 |

### 📈 Numeric Overview
|                     |   count |         mean |         std |         min |             25% |            50% |            75% |        max |
|:--------------------|--------:|-------------:|------------:|------------:|----------------:|---------------:|---------------:|-----------:|
| norad_id            |   33358 | 42670.6      | 18778.5     |    5        | 28912.2         | 44884.5        | 59505.8        |  68379     |
| period_minutes      |   33358 |   239.702    |   699.242   |    9        |    94.32        |    99.475      |   108.957      |  55699.4   |
| perigee_km          |   33358 |  3147.13     |  8978.02    |  102        |   482           |   620          |   905.75       | 314973     |
| apogee_km           |   33358 |  5810.63     | 15915.7     |  150        |   493           |   783          |  1299          | 641287     |
| inclination_degrees |   33358 |    66.3611   |    28.2713  |    0        |    50           |    70          |    97.38       |    144.64  |
| eccentricity        |   33358 |     0.103849 |     4.99215 |   -0.725044 |     0.000216279 |     0.00138111 |     0.00880932 |    575     |
| semi_major_axis_km  |   33358 | 10848.2      | 11383       | 1433.25     |  6863.77        |  7111.64       |  7556.69       | 483126     |
| launch_mass_kg      |    5505 |   878.035    |  6251.37    |    1        |   227           |   260          |   290          | 450000     |
| proxy_mass_kg       |   33358 |   444.121    |  2591.27    |    1        |    50           |   290          |   355          | 450000     |
| dry_mass_kg         |   33358 |   378.301    |  2379.22    |    0.945946 |    50           |   245.946      |   319.5        | 420000     |
| power_watts         |    5505 |  1228.67     |  3220.29    |    0        |   120           |   120          |   815          |  84000     |
| proxy_power_watts   |   33358 |   281.143    |  1380.12    |    0        |     0           |   120          |   163.846      |  84000     |
| rcs                 |   33358 |     1.82635  |     9.444   |    0.0001   |     0.0181      |     1          |     1          |    830.035 |

# Table: ucs_details

**Dimensions:** 33,358 rows × 7 columns

**Memory Footprint:** 4.99 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **lifetime_years** | `float64` | 27,853 | 16.5% | ⚠️ |
| **sat_age_years** | `float64` | 0 | 100.0% | ✅ |
| **primary_purpose** | `object` | 12,851 | 61.5% | ⚠️ |
| **detailed_purpose** | `object` | 27,853 | 16.5% | ⚠️ |
| **geo_longitude** | `float64` | 0 | 100.0% | ✅ |
| **un_registry** | `object` | 12,851 | 61.5% | ⚠️ |

### 📝 Object Overview
|                  |   count |   unique | top            |   freq |
|:-----------------|--------:|---------:|:---------------|-------:|
| primary_purpose  |   20507 |        7 | Not Applicable |  15002 |
| detailed_purpose |    5505 |       43 | Not Specified  |   4799 |
| un_registry      |   20507 |       62 | Not Applicable |  15002 |

### 📈 Numeric Overview
|                |   count |         mean |         std |     min |     25% |     50% |     75% |   max |
|:---------------|--------:|-------------:|------------:|--------:|--------:|--------:|--------:|------:|
| norad_id       |   33358 | 42670.6      | 18778.5     |    5    | 28912.2 | 44884.5 | 59505.8 | 68379 |
| lifetime_years |    5505 |     5.63465  |     3.58569 |    0.25 |     4   |     4   |     5   |    30 |
| sat_age_years  |   33358 |    19.2817   |    19.2966  |    0    |     2   |    10   |    34   |    68 |
| geo_longitude  |   33358 |     0.388015 |    12.536   | -179.8  |     0   |     0   |     0   |   359 |

# Table: risk_assessment

**Dimensions:** 33,358 rows × 4 columns

**Memory Footprint:** 1.02 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **velocity_kms** | `float64` | 0 | 100.0% | ✅ |
| **kinetic_joules** | `float64` | 0 | 100.0% | ✅ |
| **is_zombie** | `int64` | 0 | 100.0% | ✅ |

### 📈 Numeric Overview
|                |   count |            mean |             std |         min |             25% |             50% |             75% |             max |
|:---------------|--------:|----------------:|----------------:|------------:|----------------:|----------------:|----------------:|----------------:|
| norad_id       |   33358 | 42670.6         | 18778.5         | 5           | 28912.2         | 44884.5         | 59505.8         | 68379           |
| velocity_kms   |   33358 |     6.92309     |     1.37122     | 0.908319    |     7.26278     |     7.48659     |     7.62057     |    16.6766      |
| kinetic_joules |   33358 |     8.88012e+09 |     7.37178e+10 | 2.80952e+07 |     1.38271e+09 |     7.49242e+09 |     1.03116e+10 |     1.32087e+13 |
| is_zombie      |   33358 |     0.156874    |     0.363687    | 0           |     0           |     0           |     0           |     1           |

# Table: ownership_operators

**Dimensions:** 105 rows × 10 columns

**Memory Footprint:** 0.04 MB

**Primary Key Check**: ✅ No duplicate owner_code values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **owner** | `object` | 0 | 100.0% | ✅ |
| **country_operator** | `object` | 24 | 77.1% | ⚠️ |
| **users** | `object` | 24 | 77.1% | ⚠️ |
| **is_commercial** | `int64` | 0 | 100.0% | ✅ |
| **is_government** | `int64` | 0 | 100.0% | ✅ |
| **is_military** | `int64` | 0 | 100.0% | ✅ |
| **is_civil** | `int64` | 0 | 100.0% | ✅ |
| **contractor** | `object` | 24 | 77.1% | ⚠️ |
| **contractor_country** | `object` | 24 | 77.1% | ⚠️ |

### 📝 Object Overview
|                    |   count |   unique | top                 |   freq |
|:-------------------|--------:|---------:|:--------------------|-------:|
| owner_code         |     105 |      105 | AB                  |      1 |
| owner              |     105 |      105 | AB                  |      1 |
| country_operator   |      81 |       64 | MULTINATIONAL       |      8 |
| users              |      81 |       10 | Government          |     28 |
| contractor         |      81 |       54 | Thales Alenia Space |      7 |
| contractor_country |      81 |       33 | USA                 |     23 |

### 📈 Numeric Overview
|               |   count |     mean |      std |   min |   25% |   50% |   75% |   max |
|:--------------|--------:|---------:|---------:|------:|------:|------:|------:|------:|
| is_commercial |     105 | 0.52381  | 0.501828 |     0 |     0 |     1 |     1 |     1 |
| is_government |     105 | 0.514286 | 0.502193 |     0 |     0 |     1 |     1 |     1 |
| is_military   |     105 | 0.27619  | 0.449257 |     0 |     0 |     0 |     1 |     1 |
| is_civil      |     105 | 0.27619  | 0.449257 |     0 |     0 |     0 |     1 |     1 |

# Table: launch_events

**Dimensions:** 3,939 rows × 4 columns

**Memory Footprint:** 0.69 MB

**Primary Key Check**: ✅ No duplicate launch_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **launch_id** | `object` | 0 | 100.0% | ✅ |
| **launch_date** | `object` | 0 | 100.0% | ✅ |
| **launch_year** | `int64` | 0 | 100.0% | ✅ |
| **launch_site** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| launch_id   |    3939 |     3939 | 1958-002   |      1 |
| launch_date |    3939 |     3537 | 2022-08-04 |      5 |
| launch_site |    3939 |       59 | AFETR      |    737 |

### 📈 Numeric Overview
|             |   count |    mean |     std |   min |   25% |   50% |   75% |   max |
|:------------|--------:|--------:|--------:|------:|------:|------:|------:|------:|
| launch_year |    3939 | 2002.25 | 19.0305 |  1958 |  1986 |  2005 |  2021 |  2026 |


SQLite build complete: ../data/clean/orbital_debris.db
